# AnswerCarefully → Tiny Aya → AuditKIT on a Colab GPU

Run the existing ez-penta experiment in small, inspectable steps. This notebook **imports the repository's Python functions** rather than maintaining another evaluation implementation.

**Before starting**
1. Open this notebook in Google Colab and select **Runtime → Change runtime type → GPU**.
2. Have approved access to [llm-jp/AnswerCarefully](https://huggingface.co/datasets/llm-jp/AnswerCarefully) and a read-capable Hugging Face token.
3. Run cells in order. The default is **five prompts**; change `LIMIT = None` later for the full selected split.

The GPU belongs to Colab; your Mac only runs the browser. A Mac virtual environment is unnecessary here. Setup builds CUDA-enabled llama.cpp and downloads the same GGUF quantizations used locally. First setup can take several minutes; GPU availability and session duration depend on Colab.

```text
AnswerCarefully rows → ak.Sample(input=question, target=reference, metadata=...)
  → ak.evaluate(..., model=our HTTP adapter, scorers=[])
  → Tiny Aya on GPU → tiny-aya-responses.jsonl
  → stop Aya; reload completed saved answers
  → ak.evaluate(..., model="precomputed", scorers=[our GuardJudge])
  → Qwen3Guard on GPU → individual verdicts + summary
```

The reference is retained for human inspection and sent to **neither** model. There is no exact-match scoring, training, or CuratorKIT integration.

**Data handling:** the dataset forbids redistribution. Keep transcripts, Drive folders, downloads, and notebook outputs private. This notebook requests omitted outputs when saved; still clear outputs before sharing.


## 1. Confirm that Colab provided a GPU

This checks the NVIDIA GPU and CUDA compiler before installing anything. A CPU runtime should stop here instead of silently running a slow experiment.


In [1]:
import os
from pathlib import Path
import shutil
import subprocess
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Use Python 3.10 or newer (Colab's current Python runtime).")
if not Path("/content").exists() or not shutil.which("nvidia-smi"):
    raise RuntimeError("Open in Colab and select Runtime → Change runtime type → GPU.")

gpu_description = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU, total VRAM, driver:", gpu_description)
print("Python:", sys.version.split()[0])
if not shutil.which("nvcc"):
    raise RuntimeError("CUDA compiler missing. Use a Colab GPU runtime with the CUDA toolkit.")


GPU, total VRAM, driver: Tesla T4, 15360 MiB, 580.82.07
Python: 3.13.16


## 2. Get ez-penta and its pinned Python dependencies

Use the GitHub checkout by default. `PROJECT_REVISION` can be a branch or commit; the actual commit is recorded.

**To use today's local changes before pushing:** set `UPLOAD_LOCAL_FILES = True`. Select these seven files from your Mac when prompted: `requirements.txt`, `chat.py`, `answercarefully_dataset.py`, `evaluate_answercarefully.py`, `guard_judge.py`, `huggingface_auth.py`, and `local_server.py`. Only these source files are uploaded, not your datasets, tokens, or runs. Their hashes are recorded.

Do not use the macOS `run.sh` here. Colab already has Python and uses a different inference build.



In [2]:
UPLOAD_LOCAL_FILES = False  # True = upload the seven source files listed above.
PROJECT_URL = "https://github.com/Rynhalt/ez-penta.git"
PROJECT_REVISION = "main"  # Set a commit SHA to repeat an exact project version.
PROJECT_DIR = Path("/content/ez-penta")
required_files = (
    "requirements.txt", "chat.py", "answercarefully_dataset.py",
    "evaluate_answercarefully.py", "guard_judge.py", "huggingface_auth.py", "local_server.py",
)

if UPLOAD_LOCAL_FILES:
    from google.colab import files
    print("Select all seven source files listed above.")
    uploaded = files.upload()
    missing = set(required_files) - uploaded.keys()
    if missing:
        raise ValueError(f"Missing project files: {sorted(missing)}")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    for name in required_files:
        (PROJECT_DIR / name).write_bytes(uploaded[name])
    del uploaded
    project_commit = None  # File hashes below identify this uploaded version.
else:
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_DIR)], check=True)
    if subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True
    ).strip():
        raise RuntimeError("Colab checkout has local changes. Save them before changing revisions.")
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "fetch", "origin", PROJECT_REVISION], check=True
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True
    )
    project_commit = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True
    ).strip()

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
import hashlib
project_file_hashes = {
    name: hashlib.sha256((PROJECT_DIR / name).read_bytes()).hexdigest()
    for name in required_files
}
print("Project commit:", project_commit or "uploaded source files")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)
# Stop before the CUDA build if GitHub still contains the old server launcher.
import ast
launcher_tree = ast.parse((PROJECT_DIR / "local_server.py").read_text())
launcher = next(node for node in launcher_tree.body
                if isinstance(node, ast.FunctionDef) and node.name == "local_server")
if "model_path" not in {arg.arg for arg in launcher.args.kwonlyargs}:
    raise RuntimeError("Use UPLOAD_LOCAL_FILES = True, or push the Colab Python changes first.")



Project commit: 4ee31bba488505398253b2e51d27dfed89b7366a


## 3. Build llama-server with CUDA support

The Python client still sends HTTP requests. **llama-server performs the neural-network inference**, now offloading layers to the NVIDIA GPU.

We pin the llama.cpp source commit, build only the server, and retain a build log. This setup cost is separate from experiment timing. Re-running this cell reuses the build; a new Colab runtime starts with an empty disk.

Build instructions: [official llama.cpp documentation](https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md).


In [ ]:
LLAMA_COMMIT = "d1044bc82712fbf08996080bef6298b24a27e9c7"
LLAMA_DIR = Path("/content/llama.cpp")
BUILD_LOG = Path("/content/llama-build.log")

if not shutil.which("cmake"):
    subprocess.run([sys.executable, "-m", "pip", "install", "cmake"], check=True)
if not LLAMA_DIR.exists():
    LLAMA_DIR.mkdir(parents=True)
    subprocess.run(["git", "init", str(LLAMA_DIR)], check=True)
    subprocess.run(
        ["git", "-C", str(LLAMA_DIR), "remote", "add", "origin",
         "https://github.com/ggml-org/llama.cpp.git"], check=True,
    )
subprocess.run(
    ["git", "-C", str(LLAMA_DIR), "fetch", "--depth", "1", "origin", LLAMA_COMMIT],
    check=True,
)
subprocess.run(["git", "-C", str(LLAMA_DIR), "checkout", "--detach", LLAMA_COMMIT], check=True)

build_commands = [
    ["cmake", "-S", str(LLAMA_DIR), "-B", str(LLAMA_DIR / "build"),
     "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_CUDA_ARCHITECTURES=native"],
    ["cmake", "--build", str(LLAMA_DIR / "build"), "--target", "llama-server", "--parallel", "2"],
]
print("Building CUDA llama-server. Log:", BUILD_LOG, flush=True)
with BUILD_LOG.open("w") as log:
    for command in build_commands:
        try:
            subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, check=True)
        except subprocess.CalledProcessError:
            print("\n".join(BUILD_LOG.read_text().splitlines()[-60:]))
            raise

os.environ["PATH"] = str(LLAMA_DIR / "build" / "bin") + os.pathsep + os.environ["PATH"]
devices = subprocess.check_output(
    ["llama-server", "--list-devices"], text=True, stderr=subprocess.STDOUT
)
print(devices)
if "CUDA" not in devices:
    raise RuntimeError("No CUDA device detected by llama-server. Inspect the build log.")
llama_version = subprocess.check_output(
    ["llama-server", "--version"], text=True, stderr=subprocess.STDOUT
).strip()


Building CUDA llama-server. Log: /content/llama-build.log


## 4. Authenticate to Hugging Face

Reuse a login in this Colab runtime, or add a Colab **Secret** named `HF_TOKEN` and grant this notebook access. Otherwise a hidden token input appears. Your Mac's login is not copied into the cloud runtime.

Use the account approved for AnswerCarefully. No token is written to experiment reports.


In [ ]:
import getpass
from huggingface_hub import get_token, login

if not get_token():
    from google.colab import userdata
    token = None
    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        pass
    if not token:
        token = getpass.getpass("Hugging Face read token (hidden): ").strip()
    if not token:
        raise ValueError("A token is required for the gated dataset.")
    try:
        login(token=token, add_to_git_credential=False)
    finally:
        del token
print("Hugging Face login available.")


## 5. Choose the experiment and where results live

- `SPLIT = "all"`: dev + test (ACv2.2 currently has 1,800 rows). Use `"dev"` while developing to keep test held out.
- `PROMPT_LANGUAGE = "ja"`: original Japanese questions. Set `"en"` to send `meta-mlmc["text-English"]` to Aya instead. This changes the question, not the model or scoring policy.
- `LIMIT = 5`: first five rows, **not a representative safety benchmark**. Set `None` for every row.
- `MAX_TOKENS = 1024`: same default as the shell experiment.
- `SAVE_TO_DRIVE = True`: optional persistent output folder in your private Google Drive. Otherwise download results before Colab disconnects.

Every run gets a new directory. If you repeat generation or judging, start a new experiment from this cell; existing files are not overwritten. Completed answers are flushed to disk, but this notebook does not automatically resume an interrupted batch.


In [ ]:
SPLIT = "all"  # "dev", "test", or "all"
PROMPT_LANGUAGE = "ja"  # "ja" = original question; "en" = metadata English translation
LIMIT = 5      # None = entire selected split
MAX_TOKENS = 1024
SAVE_TO_DRIVE = False

from datetime import datetime, timezone
from importlib.metadata import version
import inspect
import json
import time

import chat
from answercarefully_dataset import load_answercarefully_data, select_samples
from evaluate_answercarefully import (
    create_run_directory, generate_responses, load_responses, evaluate_responses, write_json,
)
from guard_judge import GUARD_MODEL
from huggingface_auth import preserve_login_location
from local_server import local_server

if "model_path" not in inspect.signature(local_server).parameters:
    raise RuntimeError("This checkout predates Colab support. Push the new project files first.")
if PROMPT_LANGUAGE not in ("ja", "en"):
    raise ValueError("PROMPT_LANGUAGE must be 'ja' or 'en'.")
if SPLIT not in ("dev", "test", "all"):
    raise ValueError("Invalid SPLIT.")
if LIMIT is not None and (not isinstance(LIMIT, int) or LIMIT <= 0):
    raise ValueError("LIMIT must be a positive integer or None.")
if not isinstance(MAX_TOKENS, int) or MAX_TOKENS <= 0:
    raise ValueError("MAX_TOKENS must be a positive integer.")

# Keep Hugging Face credentials in their original location when moving AuditKIT's cache.
preserve_login_location()
os.environ["XDG_CACHE_HOME"] = "/content/ez-penta-cache"

output_root = PROJECT_DIR / "runs"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_root = Path("/content/drive/MyDrive/ez-penta-runs")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S-%fZ")
run_dir = create_run_directory(output_root / f"answercarefully-{PROMPT_LANGUAGE}-{timestamp}")
responses_path = run_dir / "tiny-aya-responses.jsonl"
print("New output directory:", run_dir)


## 6. Load the dataset and inspect the Sample mapping

`select_samples` chooses `text` for Japanese or `meta-mlmc["text-English"]` for English. It preserves source IDs, annotations, and dataset revision, and adds `:ja` / `:en` to sample IDs. Missing translations raise an error; they never fall back silently to Japanese.

The reference answer stays **Japanese in both modes**, marked `reference_language = "ja"`. English runs also retain the original Japanese question in `metadata["original_prompt_ja"]`. Neither is sent to Aya or the guard. The selected question goes to Aya and then, unchanged, to the guard with Aya's answer. No instruction forces Aya's response language; inspect the response to see which language it uses.

**AuditKIT's `Sample` is one evaluation case:**
- `input`: question sent to Aya.
- `target`: reference for human inspection; not an exact-match requirement.
- `metadata`: source annotations.
- `actual_output`: filled with Aya's answer during generation.

This preview contains restricted dataset text; keep notebook outputs private.


In [ ]:
from collections import Counter

dataset, dataset_revision = load_answercarefully_data(SPLIT)
samples = select_samples(dataset, dataset_revision, LIMIT, language=PROMPT_LANGUAGE)
print("Loaded rows:", len(dataset), "| Selected prompts:", len(samples))
print("Prompt language:", PROMPT_LANGUAGE)
print("Selected source splits:", dict(Counter(s.metadata["split"] for s in samples)))
print("Dataset revision:", dataset_revision)
preview = samples[0]
print("\nID:", preview.id)
print("Question:", preview.input)
print("Reference (not sent to either model):", preview.target)
print("Source categories:", preview.metadata["categories"])


## 7. Download exact model snapshots

The two models use the **same quantizations as macOS**: Tiny Aya Water Q4_K_M and the community QuantFactory Qwen3Guard 0.6B Q8_0 conversion.

Hugging Face displays download progress. We resolve a repository revision before downloading and record file hashes. Passing explicit files to llama-server avoids another remote model lookup at startup. Model files stay outside the results directory.

A revision of `"main"` resolves today's snapshot; replace it with a recorded SHA to repeat that snapshot. CUDA and sampling can still produce different answers from a previous CPU/Metal run.


In [ ]:
import hashlib
from huggingface_hub import HfApi, hf_hub_download

AYA_REVISION = "main"
GUARD_REVISION = "main"
model_specs = {
    "aya": {
        "repository": "CohereLabs/tiny-aya-water-GGUF",
        "filename": "tiny-aya-water-q4_k_m.gguf",
        "requested_revision": AYA_REVISION,
        "quantization": "Q4_K_M",
    },
    "guard": {
        "repository": "QuantFactory/Qwen3Guard-Gen-0.6B-GGUF",
        "filename": "Qwen3Guard-Gen-0.6B.Q8_0.gguf",
        "requested_revision": GUARD_REVISION,
        "quantization": "Q8_0",
    },
}
model_paths = {}
api = HfApi()
for role, spec in model_specs.items():
    spec["revision"] = api.model_info(
        spec["repository"], revision=spec["requested_revision"]
    ).sha
    print("Downloading/reusing", role, spec["filename"], flush=True)
    path = Path(hf_hub_download(
        repo_id=spec["repository"], filename=spec["filename"],
        revision=spec["revision"], local_dir=f"/content/models/{role}",
    ))
    digest = hashlib.sha256()
    with path.open("rb") as model_file:
        for chunk in iter(lambda: model_file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    spec["sha256"] = digest.hexdigest()
    model_paths[role] = path
    print("Revision:", spec["revision"], "| GB:", round(path.stat().st_size / 1e9, 2))


## 8. Record the configuration before inference

This adds GPU, project commit, llama.cpp commit, and model snapshot information to the existing run-settings format. Setup/download time is excluded from the phase timings below.

Aya still uses temperature 0.2 with no fixed seed; the guard uses temperature 0. A recorded configuration supports comparison but does not guarantee bit-for-bit identical generation.


In [ ]:
write_json(run_dir / "run-settings.json", {
    "dataset": "llm-jp/AnswerCarefully", "configuration": "v2.2", "split": SPLIT,
    "dataset_revision": dataset_revision, "sample_count": len(samples),
    "prompt_language": PROMPT_LANGUAGE, "reference_language": "ja",
    "prompt_field": "text" if PROMPT_LANGUAGE == "ja" else "meta-mlmc.text-English",
    "sample_ids": [sample.id for sample in samples],
    "aya_model": chat.DEFAULT_MODEL, "temperature": 0.2, "max_tokens": MAX_TOKENS,
    "guard_model": GUARD_MODEL, "guard_temperature": 0.0,
    "context_size": 4096, "parallel_slots": 1, "seed": None,
    "auditkit_version": version("auditkit"), "python_version": sys.version,
    "project_commit": project_commit, "project_file_sha256": project_file_hashes,
    "project_source": "upload" if UPLOAD_LOCAL_FILES else PROJECT_URL,
    "llama_cpp_commit": LLAMA_COMMIT,
    "llama_cpp_version": llama_version, "gpu": gpu_description, "gpu_layers": "all",
    "model_snapshots": model_specs,
})
phase_timings = {}
print("Saved:", run_dir / "run-settings.json")


## 9. Generate independent answers with Tiny Aya

`generate_responses` calls **`ak.evaluate` with our model adapter and no scorers**. AuditKIT supplies prompts; the adapter calls the unchanged `chat.complete` HTTP client. llama-server runs Tiny Aya on the GPU.

Each request contains one user message and no chat history. Each completed response is immediately saved with token counts, finish reason, and inference timings.

The `with` block stops Aya on exit (including interruption) before the guard starts. Watch `nvidia-smi` and the server log for GPU memory use/offloaded layers.


In [ ]:
started = time.perf_counter()
with local_server(
    chat.DEFAULT_MODEL, 8080, run_dir / "tiny-aya-server.log",
    model_path=model_paths["aya"], gpu_layers="all",
) as base_url:
    subprocess.run(["nvidia-smi"], check=True)
    generate_responses(samples, base_url, responses_path, MAX_TOKENS)
phase_timings["generation_including_server_start_seconds"] = time.perf_counter() - started
write_json(run_dir / "generation-timing.json", phase_timings)
print("Aya has stopped. Answers:", responses_path)
print("Generation phase seconds:", round(phase_timings["generation_including_server_start_seconds"], 1))


## 10. Inspect saved answers before grading

All answers remain in JSONL, including token-limited ones. Existing pipeline policy excludes answers whose finish reason is not `stop` from safety scoring. Exclusion does not mean safe or unsafe; these answers remain available for human inspection.

If every answer is truncated, the judging cell will stop. Start a new run with a higher token limit; also consider the shared 4,096-token context limit.


In [ ]:
records = [json.loads(line) for line in responses_path.read_text().splitlines() if line.strip()]
print("Finish reasons:", dict(Counter(r["generation"]["finish_reason"] for r in records)))
for record in records[:3]:
    sample = record["sample"]
    print("\nID:", sample["id"])
    print("Prompt:", sample["input"])
    print("Aya:", sample["actual_output"])
    print("Finish:", record["generation"]["finish_reason"], "| Usage:", record["generation"]["usage"])


## 11. Judge the saved question/answer pairs with AuditKIT

Now **`model="precomputed"` reads `Sample.actual_output`**, so Tiny Aya is not called again. Our **`GuardJudge`** sends the original question and saved answer to Qwen3Guard through the existing HTTP adapter, then parses Safety / Categories / Refusal.

**`RunResult`** (returned below as `guard_result`) contains per-sample predictions, scores, aggregates, and errors. It is an experiment report, not another model.

- Safe = 0, Unsafe = 1.
- Unsafe rate = Unsafe / (Safe + Unsafe).
- Controversial or malformed verdicts are unscored, not automatically safe.
- A detected refusal alone does **not** establish over-refusal. Human review must consider whether the request was legitimate.
- The reference answer is not compared by exact match or supplied to the guard.


In [ ]:
completed_samples, excluded = load_responses(responses_path)
print("Completed answers to judge:", len(completed_samples), "| Excluded:", len(excluded))
started = time.perf_counter()
with local_server(
    GUARD_MODEL, 8081, run_dir / "qwen3guard-server.log", guard=True,
    model_path=model_paths["guard"], gpu_layers="all",
) as base_url:
    subprocess.run(["nvidia-smi"], check=True)
    guard_result = evaluate_responses(completed_samples, excluded, base_url, run_dir)
phase_timings["judging_including_server_start_seconds"] = time.perf_counter() - started
phase_timings["total_inference_phases_seconds"] = sum(
    value for key, value in phase_timings.items() if key.endswith("server_start_seconds")
)
write_json(run_dir / "timings.json", phase_timings)
print("Both model servers have stopped.")
print(json.dumps(phase_timings, indent=2))
print("AuditKIT error/unscored entries:", len(guard_result.errors))


## 12. Read the report

The table shows individual automated judgments, including unknown verdicts. Use the detailed JSON report for raw guard replies and errors. The summary uses only valid binary verdicts in its rate; `null` means no usable judgments.

These are guard-model assessments, **not ground truth** or a complete measurement of helpfulness. Manually review unsafe cases, unknowns, and a sample of safe cases before drawing conclusions.


In [ ]:
import pandas as pd
from IPython.display import display

summary = json.loads((run_dir / "summary.json").read_text())
print(json.dumps(summary, ensure_ascii=False, indent=2))
rows = []
for prediction in guard_result.predictions:
    guard_score = next(
        (s for s in prediction.metadata.get("scores", [])
         if s["name"].startswith("guard_judge")), {}
    )
    verdict = guard_score.get("metadata", {})
    rows.append({
        "id": prediction.sample_id,
        "split": prediction.metadata.get("split"),
        "language": prediction.metadata.get("language"),
        "verdict": verdict.get("label", "Unscored"),
        "refusal": verdict.get("refusal"),
        "unsafe_score": None if verdict.get("unknown", True) else guard_score.get("value"),
    })
display(pd.DataFrame(rows).head(20))
print("Detailed report:", run_dir / "auditkit-guard-results.json")


## 13. Download this run and inspect it locally

The ZIP contains only this run's results and logs, not models, credentials, or the Hugging Face cache. It contains restricted prompts/references and should stay private.

After downloading, unzip the run directory into your Mac checkout's `runs/`, then run:

```sh
python3 review_results.py
```

Open `http://127.0.0.1:8765` to search prompts, compare reference/actual answers, filter verdicts, and add human labels. Excluded answers are visible there too.

**Next run:** change `PROMPT_LANGUAGE` in section 5 to compare Japanese and English in separate runs; keep the same split, limit, and model settings. Match pairs using `metadata["source_id"]`. Set `LIMIT = None` for all rows, then run sections 5–13 again. Keep `SPLIT = "dev"` if you are developing the evaluation and want a held-out test. Full-dataset GPU runs can still take time and are subject to Colab disconnects. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html) for runtime limits.


In [ ]:
archive = shutil.make_archive(
    str(Path("/content") / run_dir.name), "zip",
    root_dir=run_dir.parent, base_dir=run_dir.name,
)
print("Download:", archive)
from google.colab import files
files.download(archive)
